# TP 2 - Préparation de la base RAG


Ce notebook prépare une base documentaire simple pour le RAG, sans optimisation.
On construit ici une version de base (V1) comme point de référence.

### 0.1. Objectif
- **TP 2_1** : Créer une base de données vectorielle (`chroma_db_rag_v1`) à partir de documents Markdown issus de guides de voyage
- **TP 2_2** : Créer un assistant de voyage RAG simple
- **TP 2_3** : Créer une base de données mieux structurée / optimisée (`chroma_db_rag_v2`)
- **TP 2_4** : Créer un assistant de voyage RAG avec des méthodes avancées de retrieval

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
import shutil
from pathlib import Path

import chromadb
from tqdm import tqdm

from shared.config import ROOT_DIR, genai_client, project_settings
from shared.rag_utils import (
    CHROMA_MAX_BATCH_SIZE,
    MarkdownDocument,
    RAGChunk,
    rag_describe_chunks,
    rag_load_markdown_documents,
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
MARKDOWN_DIR = DATA_DIR / "guides_markdown"
CHROMA_DIR_V1 = DATA_DIR / "chroma_db_rag_v1"

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fonctions et classes à utiliser**

- `rag_load_markdown_documents` : charge les fichiers `.md` d'un dossier en `MarkdownDocument`
- `rag_describe_chunks` : affiche des statistiques et la distribution des tailles de chunks
- `CHROMA_MAX_BATCH_SIZE` : taille maximale d'un batch d'ajout Chroma

--> Disponibles dans `shared/rag_utils.py`

**Fonctions corrigées dans ce notebook**

- `rag_chunk_document_by_chars` : découpe un document en chunks de taille fixe avec overlap
- `rag_embed_text_batch` : calcule les embeddings d'une liste de textes
- `rag_embed_all_chunks` : calcule les embeddings de tous les chunks, en appelant `rag_embed_text_batch` par groupes
- `rag_index_chunks_chroma` : indexe des chunks vectorisés dans Chroma

--> À implémenter ici, puis à copier dans `shared/rag_utils.py`

### 0.4. Charger les documents

Utilisez la fonction `rag_load_markdown_documents` importée depuis `shared/rag_utils.py`.

Elle lit les fichiers `.md` d'un dossier et retourne une liste de `MarkdownDocument` (défini aussi dans le même fichier).

In [ ]:
documents = rag_load_markdown_documents(MARKDOWN_DIR)

total_documents = len(documents)
total_characters = sum(len(document["text"]) for document in documents)

print(f"Documents Markdown chargés : {total_documents}")
print(f"Nombre total de caractères : {total_characters}")

---
## 1. Stratégie de chunking classique (V1)

Un document complet est **trop long** pour la recherche par vecteurs latents.
On le **découpe en chunks** de taille comparable, mesurés en nombre de caractères.
L'**overlap** garde une zone de texte partagée entre deux chunks voisins pour conserver la continuité.

Repère attendu : entre 100 et 300 chunks au total.

### 1.1. Fonction de chunking corrigée

In [ ]:
def rag_chunk_document_by_chars(
    document: MarkdownDocument,
    chunk_chars: int,
    chunk_overlap_chars: int,
) -> list[RAGChunk]:
    """Découper un document en chunks de taille fixe avec fenêtre glissante

    Entrées
    - document : un `MarkdownDocument`
    - chunk_chars : taille max d'un chunk en caractères
    - chunk_overlap_chars : chevauchement entre deux chunks successifs

    Sortie
    - liste de `RAGChunk`
    """
    chunks: list[RAGChunk] = []
    step = chunk_chars - chunk_overlap_chars
    chunk_id = 0

    for start_index in range(0, len(document["text"]), step):
        chunk_text = document["text"][start_index:start_index + chunk_chars].strip()
        if chunk_text:
            chunks.append(RAGChunk(source=document["source"], chunk_id=chunk_id, text=chunk_text))
            chunk_id += 1

    return chunks

### 1.2. Appliquer le chunking à tous les documents

In [ ]:
chunks_v1 = []
for doc in documents:
    chunks_v1.extend(rag_chunk_document_by_chars(doc, chunk_chars=3000, chunk_overlap_chars=250))

### 1.3. Inspecter les chunks (statistiques)

`rag_describe_chunks` affiche des statistiques (nombre de chunks, taille min/max/moyenne) et un histogramme de la distribution des tailles par document source.

In [ ]:
rag_describe_chunks(chunks_v1)

### 1.4. Afficher quelques chunks

Afficher quelques chunks pour comprendre le résultat du découpage.

In [ ]:
sample_indices = [0, len(chunks_v1) // 4, len(chunks_v1) // 2, 3 * len(chunks_v1) // 4, len(chunks_v1) - 1]

for idx in sample_indices:
    chunk = chunks_v1[idx]
    print(f"-- Chunk #{chunk.chunk_id} | source: {chunk.source} | {len(chunk.text)} caractères ---")
    print(chunk.text[:500])
    print("..." if len(chunk.text) > 500 else "")
    print("\n\n\n")

---
## 2. Embeddings et indexation

Dans cette partie, nous allons calculer les embeddings et les indexer dans une base de données Chroma.

**Étape 1 : Calcul des embeddings** - Gemini API

- `rag_embed_text_batch` calcule les embeddings d'un lot (*batch*) de textes (l'API a une limite de textes à envoyer en un seul appel)
- `rag_embed_all_chunks` appelle `rag_embed_text_batch` autant de fois que nécessaire pour calculer les embeddings de tous les chunks (par batch)

**Étape 2 : Indexation des embeddings** - Chroma

- `rag_index_chunks_chroma` indexe les embeddings dans une base de données Chroma (SQLite)

### 2.1. Fonction d'embedding corrigée

In [ ]:
def rag_embed_text_batch(texts: list[str]) -> list[list[float]]:
    """Calculer les embeddings d'une liste de textes via l'API Google GenAI

    Entrées
    - texts : liste de chaînes à vectoriser

    Sortie
    - liste de vecteurs `list[float]` dans le même ordre que `texts`
    """
    response = genai_client.models.embed_content(
        model=project_settings.rag_embedding_model_name,
        contents=texts,
    )
    return [embedding.values for embedding in response.embeddings]

### 2.2. Fonction de batch embedding corrigée

In [ ]:
def rag_embed_all_chunks(chunks: list[RAGChunk], batch_size: int = 16) -> list[RAGChunk]:
    """Calculer les embeddings de tous les chunks par batch

    Entrées
    - chunks : liste de `RAGChunk`
    - batch_size : nombre de chunks traités par appel embedding

    Sortie
    - liste de `RAGChunk` avec `embedding` renseigné
    """
    embedded_chunks: list[RAGChunk] = []

    for start_index in tqdm(range(0, len(chunks), batch_size), desc="Calcul des embeddings", unit="batch"):
        batch = chunks[start_index:start_index + batch_size]
        batch_texts = [f"passage: {chunk.text}" for chunk in batch]
        batch_vectors = rag_embed_text_batch(batch_texts)

        for chunk, vector in zip(batch, batch_vectors):
            embedded_chunks.append(
                RAGChunk(
                    source=chunk.source,
                    chunk_id=chunk.chunk_id,
                    text=chunk.text,
                    embedding=vector,
                )
            )

    return embedded_chunks

### 2.3. Calculer les embeddings des chunks

In [ ]:
chunk_embeddings_v1 = rag_embed_all_chunks(
    chunks=chunks_v1,
    batch_size=16
)

### 2.4. Fonction d'indexation corrigée

In [ ]:
def rag_index_chunks_chroma(persist_dir: Path, chunks: list[RAGChunk]) -> None:
    """Indexer des chunks vectorisés dans Chroma

    Entrées
    - persist_dir : dossier de persistance Chroma
    - chunks : liste de `RAGChunk` avec embeddings calculés

    Sortie
    - None
    """
    if persist_dir.exists():
        shutil.rmtree(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    client = chromadb.PersistentClient(path=str(persist_dir))
    collection = client.get_or_create_collection(name="chunks")

    ids = [f"{chunk.source}_{chunk.chunk_id}" for chunk in chunks]
    docs = [chunk.text for chunk in chunks]
    embeddings = [chunk.embedding for chunk in chunks]
    metadatas = [{"source": chunk.source, "chunk_id": str(chunk.chunk_id)} for chunk in chunks]

    total = len(ids)
    for start in range(0, total, CHROMA_MAX_BATCH_SIZE):
        end = min(start + CHROMA_MAX_BATCH_SIZE, total)
        collection.add(
            ids=ids[start:end],
            documents=docs[start:end],
            embeddings=embeddings[start:end],
            metadatas=metadatas[start:end],
        )

### 2.5. Indexer les chunks dans Chroma

In [ ]:
rag_index_chunks_chroma(persist_dir=CHROMA_DIR_V1, chunks=chunk_embeddings_v1)

print(f"V1 - Chunks indexés : {len(chunk_embeddings_v1)}")
print(f"V1 - Dimension des embeddings : {len(chunk_embeddings_v1[0].embedding)}")